# MindSpore 中文情感分类（ChnSentiCorp + TextCNN）

**选题 A** · Windows CPU · MindSpore 2.10.0 · **无教学视频**

本 notebook 端到端完成：下载数据 → 字级词表 → 训练 TextCNN → 评估准确率 → 导出 demo 图。

目标：测试准确率 **≥ 80%**。

## 0. 环境检查

请在 conda 环境 `ms_docs_try` 中打开本文件。若 `import mindspore` 失败，先按仓库根目录 `README.md` 安装。

In [ ]:
import sys
from pathlib import Path

# 保证能 import 仓库根下的 src/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import mindspore as ms
print("MindSpore:", ms.__version__)
print("Python:", sys.version.split()[0])
print("ROOT:", ROOT)

# 强制 CPU；GRAPH_MODE 更利于 Windows/Linux CPU 训练速度
ms.set_device("CPU")
ms.set_context(mode=ms.GRAPH_MODE)
print("device: CPU, mode: GRAPH")

## 1. 配置超参

默认 TextCNN + 字级编码。完整 ChnSentiCorp 上 5 epoch 通常可过 80%。若机器较慢，可先把 `EPOCHS` 设为 2 做冒烟，再改回 5～8。

In [ ]:
from dataclasses import dataclass

@dataclass
class Config:
    model_name: str = "textcnn"   # textcnn | bilstm
    max_len: int = 100            # CPU 友好；可提到 128
    max_vocab: int = 8000
    embed_dim: int = 128
    batch_size: int = 64
    epochs: int = 5               # 冲准确率可改为 8～10
    lr: float = 1e-3
    seed: int = 42
    test_ratio: float = 0.2

cfg = Config()
print(cfg)

## 2. 下载并准备 ChnSentiCorp

数据来自 [ChineseNlpCorpus](https://github.com/SophonPlus/ChineseNlpCorpus) 的酒店评论全集。列：`label`（0/1）、`review`。

**为何字级分词？** Windows CPU 作业希望依赖少；字级 TextCNN 对中文情感足够达到 ≥80%。

In [ ]:
from src.data import (
    build_vocab,
    ensure_chnsenticorp,
    load_reviews,
    prepare_arrays,
    save_vocab,
    train_test_split,
)

csv_path = ensure_chnsenticorp(ROOT / "data")
samples = load_reviews(csv_path)
print(f"有效样本数: {len(samples)}")
print("样例:", samples[0][0][:60], "... label=", samples[0][1])

train_samples, test_samples = train_test_split(
    samples, test_ratio=cfg.test_ratio, seed=cfg.seed
)
pos = sum(1 for _, y in train_samples if y == 1)
neg = len(train_samples) - pos
print(f"训练集: {len(train_samples)} (正 {pos} / 负 {neg})")
print(f"测试集: {len(test_samples)}")

vocab = build_vocab(
    [t for t, _ in train_samples],
    max_vocab_size=cfg.max_vocab,
    min_freq=2,
)
(ROOT / "checkpoints").mkdir(parents=True, exist_ok=True)
save_vocab(vocab, ROOT / "checkpoints" / "vocab.txt")
print(f"词表大小: {len(vocab)}")

train_xs, train_ys = prepare_arrays(train_samples, vocab, max_len=cfg.max_len)
test_xs, test_ys = prepare_arrays(test_samples, vocab, max_len=cfg.max_len)
print("batch 向量长度:", len(train_xs[0]))

## 3. 构建模型

**TextCNN**：多尺度卷积核 (3,4,5) + max-over-time pooling，适合短评论。  
可选 BiLSTM 作对比（更慢，一般也能过线）。

In [ ]:
from src.model import build_model
from src.train import setup_mindspore

setup_mindspore(device="CPU", seed=cfg.seed)
model, model_name = build_model(
    cfg.model_name, vocab_size=len(vocab), embed_dim=cfg.embed_dim
)
print("使用模型:", model_name)
# 打印参数量（大约）
n_params = sum(p.size for p in model.trainable_params())
print(f"可训练参数量: {n_params:,}")

## 4. 训练

每个 epoch 后在测试集评估；最优权重写入 `checkpoints/`。

In [ ]:
from src.train import fit

ckpt_path = ROOT / "checkpoints" / f"{cfg.model_name}_best.ckpt"
history = fit(
    model,
    train_xs,
    train_ys,
    test_xs,
    test_ys,
    epochs=cfg.epochs,
    batch_size=cfg.batch_size,
    lr=cfg.lr,
    ckpt_path=ckpt_path,
)
print("历史 test_acc:", ["%.4f" % a for a in history["test_acc"]])

## 5. 最终评估与达标判断

In [ ]:
from mindspore import nn
import numpy as np
from src.train import evaluate, load_weights

# 加载最优 checkpoint 再评一次
if ckpt_path.exists():
    load_weights(model, ckpt_path)
    print("已加载:", ckpt_path)

loss_fn = nn.SoftmaxCrossEntropyWithLogits(sparse=True, reduction="mean")
x_test = np.asarray(test_xs, dtype=np.int32)
y_test = np.asarray(test_ys, dtype=np.int32)
test_loss, test_acc, y_true, y_pred = evaluate(
    model, x_test, y_test, loss_fn, batch_size=cfg.batch_size
)
print(f"测试 loss={test_loss:.4f}  acc={test_acc:.4f}")

if test_acc >= 0.80:
    print("✅ 已达到作业门槛：准确率 ≥ 80%")
else:
    print("⚠️ 未达 80%。建议：增大 EPOCHS 到 8～10，或 embed_dim=192 后重跑本节以上单元格。")

## 6. 导出 Demo 图（提交 GitHub / Gallery 用）

In [ ]:
from src.viz import plot_confusion, plot_history, plot_sample_predictions

demo_dir = ROOT / "assets" / "demo"
demo_dir.mkdir(parents=True, exist_ok=True)

p1 = plot_history(history, demo_dir / "accuracy_loss_curves.png")
p2 = plot_confusion(y_true, y_pred, demo_dir / "confusion_matrix.png")

show = []
for (text, y_t), y_p in zip(test_samples, y_pred):
    show.append((text, int(y_t), int(y_p)))
    if len(show) >= 12:
        break
p3 = plot_sample_predictions(show, demo_dir / "sample_predictions.png")

print("已保存:")
for p in (p1, p2, p3):
    print(" -", p)

## 7.（可选）简单 Baseline 对比

用「训练集多数类」作弱 baseline，证明 TextCNN 有明显提升（若准确率卡线，可写进 README 的 before/after）。

In [ ]:
import numpy as np

majority = int(round(np.mean(train_ys)))
baseline_acc = float(np.mean(np.array(test_ys) == majority))
print(f"多数类 baseline 准确率: {baseline_acc:.4f} (恒预测 {majority})")
print(f"TextCNN 准确率:          {test_acc:.4f}")
print(f"相对提升:                {test_acc - baseline_acc:+.4f}")

## 8. 单条推理 Demo

In [ ]:
from src.data import encode
import numpy as np
from mindspore import Tensor

def predict_sentiment(text: str) -> str:
    ids = encode(text, vocab, max_len=cfg.max_len)
    x = Tensor(np.array([ids], dtype=np.int32))
    model.set_train(False)
    logits = model(x)
    pred = int(logits.argmax(axis=1).asnumpy()[0])
    return "正向" if pred == 1 else "负向"

demos = [
    "房间干净整洁，服务态度很好，下次还来！",
    "卫生太差了，隔音不好，不会再住。",
    "位置方便，早餐一般，总体还行。",
]
for t in demos:
    print(predict_sentiment(t), "←", t)

## 完成检查

- [ ] 测试准确率 ≥ 80%（或 README 写清相对 baseline 提升）
- [ ] `assets/demo/` 三张图已生成
- [ ] 推送 GitHub；单 notebook 验证后上传 AI Gallery
- [ ] **不做**教学视频

本地逐步命令见训练营文档 `camp-topic-a-runbook.md`。